In [ ]:
import pandas as pd
DATA_DIR = r"C:\Users\HARDPC\Desktop\AL\projekty\products_prices_historical_analysis\keepa_data"

In [ ]:
meta = pd.read_csv(DATA_DIR + r"\all_products_meta.csv")

meta.shape
meta.info()

KEEPA_EPOCH = pd.Timestamp("2011-01-01")
meta['listed_since'] = KEEPA_EPOCH + pd.to_timedelta(meta['listedSince'], unit = 'min')
meta['tracking_since'] = KEEPA_EPOCH + pd.to_timedelta(meta['trackingSince'], unit = 'min')

meta["tracking_gap_days"] = (meta["tracking_since"] - meta["listed_since"]).dt.days
meta[["asin", "brand", "model", "listed_since", "tracking_since", "tracking_gap_days"]].sort_values("tracking_gap_days", ascending=False)


In [ ]:
meta.info()
KEEPA_EPOCH = pd.Timestamp("2011-01-01")
meta['listed_since'] = KEEPA_EPOCH + pd.to_timedelta(meta['listedSince'], unit = 'min')
meta['tracking_since'] = KEEPA_EPOCH + pd.to_timedelta(meta['trackingSince'], unit = 'min')
meta["tracking_gap_days"] = (meta["tracking_since"] - meta["listed_since"]).dt.days
meta[["asin", "brand", "model", "listed_since", "tracking_since", "tracking_gap_days"]].sort_values("tracking_gap_days", ascending=False)


In [ ]:
ph = pd.read_csv(DATA_DIR + r"\B09LNW3CY2_price_history.csv")
ph.shape
ph.head(10)
ph.info


price_cols = ["AMAZON", "NEW", "USED", "LISTPRICE", "REFURBISHED"]
ph[price_cols].notna().sum()


In [ ]:
#price history data loading - combining all price history .csv files into one DF - os + pandas
#note - data/ files are not shared in repo, as it would obviously be very ineffective, samples in data_schema_examples.md
import os
import pandas as pd

df = pd.DataFrame()
df2 = pd.DataFrame()
data_path = os.path.join('../data/')
for file in os.listdir(data_path):
    if '_price_history' in file:
        df2 = pd.read_csv(data_path + file)
        df = pd.concat([df, df2])


In [ ]:
#data resampling - from the loaded 
filtered_df = df[df['NEW'].notna()]
filtered_df['datetime'] = pd.to_datetime(filtered_df['datetime'])
filtered_df = filtered_df.set_index('datetime', drop = True)

pre_filtered_df = filtered_df.groupby("asin").resample("W")["NEW"].min()
print(pre_filtered_df)
filtered_df = filtered_df.groupby("asin").resample("W")["NEW"].min().reset_index()
print(filtered_df)
print(filtered_df['NEW'].notna().sum()) #There are still 78 NaN values (out of 3062 values, 2.5% - it's alright), which means in case of 78 weeks in total we have no price points.
#I thought about using a mean of previous/next row, but we can also assume that the price DID NOT CHANGE during that time, so using ffill makes more sense here.

filtered_df['NEW'] = filtered_df.groupby('asin')['NEW'].ffill()
print(filtered_df['NEW'].notna().sum())
print(filtered_df.columns)
print(filtered_df.head())

In [ ]:
#W1 D3 - rows where USED is not null + convert datetime + set datetime
#T1
# combined_df = df[df['USED'].notna()]
# combined_df['datetime'] = pd.to_datetime(combined_df['datetime'])
# print(combined_df.shape)
# combined_df.head(3)
# combined_df = combined_df.set_index('datetime', drop = True)

# #T2
# resampled_df = combined_df.groupby('asin').resample('W').min().reset_index() #resmapling to get the min price for each asin for each week
# print(resampled_df['USED'].isna().sum()) #168 nulls at this point
# resampled_df['USED'] = resampled_df['USED'].ffill()
# print(resampled_df['USED'].isna().sum()) #0 nulls at this point
# print(resampled_df.head(5))
# print(resampled_df['asin'].value_counts())


#T4
meta_path = '../data/all_products_meta.csv'
meta = pd.read_csv(meta_path)
meta['listed_since'] = KEEPA_EPOCH + pd.to_timedelta(meta['listedSince'], 'minutes')
meta = meta[['asin', 'brand', 'model', 'listed_since']]

#T5 - merged meta with resampled minimum NEW prices on weekly interval
merged_df = filtered_df.merge(meta, on = 'asin') #merged meta with resampled NEW prices
print(merged_df.shape)
print(merged_df.dtypes)
print(merged_df.head())

#T6 - filtered for iphone 13 only (asin = B09LNW3CY2)
iphone13_df = merged_df[merged_df['asin'] == 'B09LNW3CY2']
print(iphone13_df)

#T7 - simple line chart - I needed to check the Plotly docs for this, as we haven't learned it before, but it was easy tbh.
import plotly.express as px
chart = px.line(iphone13_df, x='datetime', y='NEW', title = 'iPhone 13 price over time')
chart.show()

In [ ]:
#W1 D4 
#T1 - repeating the pattern from the last days - filtering, checking the nulls, setting datetime to proper dattime format,
# setting the index and resampling to weekly format

amazon_df = df[df['AMAZON'].notna()]
amazon_df['datetime'] = pd.to_datetime(amazon_df['datetime'])
amazon_df = amazon_df.set_index('datetime', drop = True)

amazon_df = amazon_df.groupby('asin').resample('W').min().reset_index()
amazon_df['AMAZON'] = amazon_df.groupby('asin')['AMAZON'].ffill()

print(amazon_df['AMAZON'].isna().sum())
amazon_df.shape
amazon_df.head


0


<bound method NDFrame.head of            asin   datetime  AMAZON     NEW    USED  SALES  LISTPRICE  \
0    B07ZPKN6YR 2023-10-22  330.53     NaN     NaN    NaN        NaN   
1    B07ZPKN6YR 2023-10-29  330.53     NaN     NaN    NaN        NaN   
2    B07ZPKN6YR 2023-11-05  330.53     NaN     NaN    NaN        NaN   
3    B07ZPKN6YR 2023-11-12  330.53     NaN     NaN    NaN        NaN   
4    B07ZPKN6YR 2023-11-19  330.53     NaN     NaN    NaN        NaN   
..          ...        ...     ...     ...     ...    ...        ...   
713  B0FG1THCD7 2026-02-08  649.90  679.79     NaN  514.0        NaN   
714  B0FG1THCD7 2026-02-15  509.99     NaN     NaN  312.0        NaN   
715  B0FG1THCD7 2026-02-22  509.99  509.99  484.49  346.0        NaN   
716  B0FG1THCD7 2026-03-01  509.99     NaN     NaN    NaN        NaN   
717  B0FG1THCD7 2026-03-08  559.99  559.99     NaN    NaN        NaN   

     REFURBISHED  COUNT_NEW  COUNT_USED  COUNT_REFURBISHED  EBAY_NEW_SHIPPING  \
0            NaN        